# ทดสอบและทำนายผลตัวอักษรไทย (Inference - Top-1)
### โครงงานจำแนกตัวอักษรและตัวเลขภาษาไทย 72 คลาส ด้วย ResNet-18


In [1]:
import os
import glob
import json
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
import torchvision.transforms.functional as TF

plt.rcParams['font.family'] = 'Tahoma'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# 1. โครงสร้างโมเดล ResNet-18 (Transfer Learning 72 คลาส)
def build_model(num_classes=72):
    model = resnet18(weights=None)
    model.fc = nn.Sequential(
        nn.Dropout(0.2),
        nn.Linear(model.fc.in_features, num_classes)
    )
    return model

# 2. โหลด Classes และ Mapping ภาษาไทย
with open('classes.json', 'r', encoding='utf-8') as f:
    classes = json.load(f)

with open('char_mapping.json', 'r', encoding='utf-8') as f:
    char_map = json.load(f)

idx_to_class = {i: c for i, c in enumerate(classes)}

# 3. โหลดค่าน้ำหนัก best_model.pt
model = build_model(num_classes=len(classes))
checkpoint = torch.load('best_model.pt', map_location=device)
sd = checkpoint.get('model_state_dict', checkpoint)
model.load_state_dict({k.replace('backbone.', ''): v for k, v in sd.items()})
model = model.to(device)
model.eval()

# 4. Data Transform (224x224)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
print('โมเดลพร้อมใช้งานแล้ว!')


## 1. ฟังก์ชันทำนายภาพเดี่ยว (Single Image Prediction)


In [ ]:
def predict_image(img_path, show=True):
    if not os.path.exists(img_path):
        print(f'ไม่พบไฟล์: {img_path}')
        return None
        
    img = Image.open(img_path).convert('RGB')
    x = transform(img).unsqueeze(0).to(device)
    
    # ทำนายผลพร้อม TTA หมุน -5, 0, 5 องศาเพื่อความแม่นยำสูงสุด
    with torch.no_grad():
        probs = torch.stack([
            torch.softmax(model(TF.rotate(x, angle, fill=1.0)), dim=1)
            for angle in [-5, 0, 5]
        ]).mean(dim=0).squeeze(0)
        
    # คำนวณเฉพาะอันดับ 1 (Top-1) เพียงอย่างเดียว
    best_idx = probs.argmax().item()
    code = idx_to_class[best_idx]
    confidence = probs[best_idx].item() * 100
    info = char_map.get(str(code), {})
    char = info.get('char', 'N/A')
    desc = info.get('description', '')
    
    print(f'ไฟล์: {os.path.basename(img_path)} -> ทาย: {char} ({desc}) | คลาส: {code} | ความมั่นใจ: {confidence:.2f}%')
    
    if show:
        plt.figure(figsize=(3.5, 3.5))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f'{char} ({desc})\nมั่นใจ {confidence:.2f}%', fontsize=12, fontweight='bold')
        plt.show()
        
    return {'code': code, 'char': char, 'description': desc, 'confidence': confidence}

# วิธีเรียกใช้งาน 
predict_image('ThaiCharacter Dataset/round2/161/bc_001sg_3_118.jpg')


## 2. ฟังก์ชันทำนายทั้งโฟลเดอร์ภาพของอาจารย์ (Directory Prediction)


In [ ]:
def predict_directory(dir_path, output_csv='inference_results.csv'):
    if not os.path.exists(dir_path):
        print(f'ไม่พบโฟลเดอร์: {dir_path}')
        return None
        
    # ดึงรายชื่อไฟล์รูปภาพในโฟลเดอร์ตรงๆ
    valid_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
    files = sorted([os.path.join(dir_path, f) for f in os.listdir(dir_path) if f.lower().endswith(valid_exts)])
    
    if not files:
        print(f'ไม่พบไฟล์รูปภาพใน: {dir_path}')
        return None
        
    print(f'พบภาพทั้งหมด {len(files)} ภาพ กำลังเริ่มทำนาย...')
    
    rows = []
    for i, fp in enumerate(files):
        res = predict_image(fp, show=False)
        rows.append({
            'filename': os.path.basename(fp),
            'predicted_code': res['code'],
            'predicted_char': res['char'],
            'description': res['description'],
            'confidence': f"{res['confidence']:.2f}%"
        })
        if (i + 1) % 50 == 0 or (i + 1) == len(files):
            print(f'ประมวลผลแล้ว [{i + 1}/{len(files)}] ภาพ...')
            
    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f'\n[สำเร็จ] บันทึกผลลัพธ์ลงในไฟล์ {output_csv} เรียบร้อยแล้ว!')
    return df

# วิธีเรียกใช้งาน
# predict_directory('path_to_teacher_test_folder/')


## 3. แสดงตารางตัวอย่างการทำนาย (Visual Demo)


In [4]:
def show_demo(folder_path, n=8):
    if not os.path.exists(folder_path):
        print(f'ไม่พบโฟลเดอร์: {folder_path}')
        return
        
    valid_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
    # ดึงไฟล์รูปภาพจากโฟลเดอร์ของอาจารย์ตรงๆ
    files = sorted([os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.lower().endswith(valid_exts)])
    
    if not files:
        print(f'ไม่พบไฟล์รูปภาพใน: {folder_path}')
        return
        
    # สุ่มเลือกภาพมา n ภาพ (หรือถ้ามีน้อยกว่า n ก็แสดงทั้งหมด)
    num_samples = min(n, len(files))
    chosen = np.random.choice(files, num_samples, replace=False)
    
    cols = 4
    rows = (num_samples + cols - 1) // cols
    plt.figure(figsize=(15, rows * 3.5))
    
    for i, fp in enumerate(chosen):
        res = predict_image(fp, show=False)
        img = Image.open(fp).convert('RGB')
        
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.axis('off')
        
        # ถ้าโฟลเดอร์มีเฉลย (ชื่อโฟลเดอร์เป็นรหัสคลาส) จะเช็คถูก/ผิดให้ด้วย
        parent = os.path.basename(os.path.dirname(fp))
        if parent in classes:
            true_char = char_map.get(str(parent), {}).get('char', parent)
            is_ok = (str(res['code']) == str(parent))
            color = 'darkgreen' if is_ok else 'crimson'
            status = '✓ ถูกต้อง' if is_ok else '✗ ผิด'
            plt.title(f'จริง: {true_char} ({parent})\nทาย: {res["char"]} [{res["confidence"]:.1f}%]\n{status}',
                      fontsize=10, fontweight='bold', color=color)
        else:
            plt.title(f'{os.path.basename(fp)}\nทาย: {res["char"]} ({res["description"]})\nมั่นใจ {res["confidence"]:.1f}%',
                      fontsize=10, fontweight='bold', color='navy')
                      
    plt.tight_layout()
    plt.show()

# ตัวอย่างการใช้งาน (ใส่โฟลเดอร์ของอาจารย์เหมือนกัน):
# show_demo('path_to_teacher_test_folder/', n=8)

# ทดสอบรันตัวอย่างกับโฟลเดอร์ในเครื่อง:
test_folder = 'ThaiCharacter Dataset/round2/161'
if os.path.exists(test_folder):
    show_demo(test_folder, n=8)
